In [3]:
%matplotlib inline
import pandas as pd
from pandas import DataFrame, Series
import numpy as np
import math
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.legend_handler import HandlerLine2D, HandlerTuple
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
#import scikit_posthocs as sp
import sys

In [4]:
AllName="dataG.pkl"
ResizesName="dataM.pkl"
matrixIt="dataL.pkl"
matrixIt_Total="data_L_Total.csv"
n_qty=8 #CAMBIAR SEGUN LA CANTIDAD DE NODOS USADOS
n_groups= 2
repet = 5 #CAMBIAR EL NUMERO SEGUN NUMERO DE EJECUCIONES POR CONFIG
time_constant = False # Cambiar segun el speedUp usado
speedup = 0.66 # Porcentaje del speedup ideal

p_value = 0.05
values = [2, 10, 20, 40, 80, 120, 160, 180]
#                      WORST          BEST
dist_names = ['null', 'BalancedFit', 'CompactFit']

processes = [2,10,20,40,80,120,160,180]

labelsP = [['(2,2)', '(2,10)', '(2,20)', '(2,40)'],['(10,2)', '(10,10)', '(10,20)', '(10,40)'],
          ['(20,2)', '(20,10)', '(20,20)', '(20,40)'],['(40,2)', '(40,10)', '(40,20)', '(40,40)']]
labelsP_J = ['(2,2)', '(2,10)', '(2,20)', '(2,40)','(10,2)', '(10,10)', '(10,20)', '(10,40)',
              '(20,2)', '(20,10)', '(20,20)', '(20,40)','(40,2)', '(40,10)', '(40,20)', '(40,40)']
positions = [321, 322, 323, 324, 325]
positions_small = [221, 222, 223, 224]

labels = ['(1,10)', '(1,20)', '(1,40)','(1,80)','(1,120)',
            '(10,1)', '(10,20)', '(10,40)','(10,80)','(10,120)',
            '(20,1)',  '(20,10)','(20,40)','(20,80)','(20,120)',
            '(40,1)',  '(40,10)',  '(40,20)','(40,80)','(40,120)',
            '(80,1)',  '(80,10)',  '(80,20)', '(80,40)','(80,120)',
            '(120,1)', '(120,10)', '(120,20)','(120,40)','(120,80)']

labelsExpand = ['(1,10)', '(1,20)', '(1,40)','(1,80)','(1,120)',
               '(10,20)', '(10,40)','(10,80)','(10,120)',
               '(20,40)','(20,80)','(20,120)',
               '(40,80)','(40,120)',
               '(80,120)']
labelsShrink = ['(10,1)', 
               '(20,1)',  '(20,10)', 
               '(40,1)',  '(40,10)',  '(40,20)',
               '(80,1)',  '(80,10)',  '(80,20)', '(80,40)',
               '(120,1)', '(120,10)', '(120,20)','(120,40)','(120,80)']

labelsExpandOrdered = ['(1,10)', '(1,20)', '(10,20)',
                       '(1,40)','(10,40)','(20,40)',
                       '(1,80)','(10,80)','(20,80)','(40,80)',
                       '(1,120)', '(10,120)', '(20,120)','(40,120)','(80,120)']
labelsShrinkOrdered = ['(10,1)', '(20,1)', '(40,1)', '(80,1)', '(120,1)',
                '(20,10)',  '(40,10)',  '(80,10)',  '(120,10)', 
                '(40,20)', '(80,20)', '(120,20)',
                '(80,40)','(120,40)',
                '(120,80)']

labelsExpandIntra = ['(1,10)', '(1,20)','(10,20)']
labelsShrinkIntra = ['(10,1)', '(20,1)', '(20,10)']
labelsExpandInter = ['(1,40)','(1,80)', '(1,160)',
               '(10,40)','(10,80)', '(10,160)',
               '(20,40)','(20,80)', '(20,160)',
               '(40,80)', '(40,160)',
               '(80,160)']
labelsShrinkInter = ['(40,1)', '(40,10)', '(40,20)',
               '(80,1)', '(80,10)', '(80,20)','(80,40)',
               '(160,1)', '(160,10)', '(160,20)','(160,40)', '(160,80)']

                #0          #1                 #2                     #3
labelsMethods = ['Baseline', 'Baseline single','Baseline - Asynchronous','Baseline single - Asynchronous',
                 'Merge','Merge single','Merge - Asynchronous','Merge single - Asynchronous']
                 #4      #5             #6                 #7
colors_spawn = ['green','springgreen','blue','darkblue','red','darkred','darkgoldenrod','olive','violet']
linestyle_spawn = ['-', '--', '-.', ':']
markers_spawn = ['.','v','s','p', 'h','d','X','P','^']

OrMult_patch = mpatches.Patch(hatch='', facecolor='green', label='Baseline')
OrSing_patch = mpatches.Patch(hatch='', facecolor='springgreen', label='Baseline single')
OrPthMult_patch = mpatches.Patch(hatch='//', facecolor='blue', label='Baseline - Asyncrhonous')
OrPthSing_patch = mpatches.Patch(hatch='\\', facecolor='darkblue', label='Baseline single - Asyncrhonous')
MergeMult_patch = mpatches.Patch(hatch='||', facecolor='red', label='Merge')
MergeSing_patch = mpatches.Patch(hatch='...', facecolor='darkred', label='Merge single')
MergePthMult_patch = mpatches.Patch(hatch='xx', facecolor='yellow', label='Merge - Asyncrhonous')
MergePthSing_patch = mpatches.Patch(hatch='++', facecolor='olive', label='Merge single - Asyncrhonous')

handles_spawn = [OrMult_patch,OrSing_patch,OrPthMult_patch,OrPthSing_patch,MergeMult_patch,MergeSing_patch,MergePthMult_patch,MergePthSing_patch]

In [14]:
dfG = pd.read_pickle( AllName )

dfG['ADR'] = (dfG['ADR'] / dfG['DR']) * 100
       
group = dfG.groupby(['Redistribution_Method', 'Redistribution_Strategy', 'Groups'])['T_total']

grouped_aggG = group.agg(['median'])
grouped_aggG.rename(columns={'median':'T_total'}, inplace=True)

TypeError: unhashable type: 'list'

In [12]:
dfG

0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
15    0.0
16    0.0
17    0.0
18    0.0
19    0.0
20    0.0
21    0.0
22    0.0
23    0.0
24    0.0
25    0.0
26    0.0
27    0.0
28    0.0
29    0.0
30    0.0
31    0.0
32    0.0
33    0.0
34    0.0
35    0.0
36    0.0
37    0.0
38    0.0
39    0.0
40    0.0
41    0.0
42    0.0
43    0.0
44    0.0
45    0.0
46    0.0
47    0.0
48    0.0
49    0.0
50    0.0
51    0.0
52    0.0
53    0.0
dtype: float64